# 03 — Robustness, efficiency, generalization and paper assets (Colab GPU)

Fills in EXP-009/010/011/012. These are **evaluation** studies that reuse the trained checkpoints — none of them retrains from scratch.

Run `02_train_and_ablate.ipynb` first: this notebook expects `results/runs/EXP-001` and `results/runs/EXP-007` to exist.

In [ ]:
import os, glob, torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
WORK = '/content/sarr-imaging'
if not os.path.isdir(WORK):
    !git clone --depth 1 https://github.com/officialarghya29/sarr-imaging.git $WORK
%cd $WORK
!pip -q install ultralytics pandas matplotlib

DATASET = 'ssdd'
IMGSZ   = 640
BASE_W  = glob.glob('results/runs/EXP-001/weights/best.pt')
OUR_W   = glob.glob('results/runs/EXP-007/weights/best.pt')
print('baseline weights:', BASE_W)
print('SAR-YOLO weights:', OUR_W)
assert OUR_W, 'Run notebook 02 first: no EXP-007 checkpoint found.'

## 1. EXP-010 — Efficiency

Params, FLOPs, latency, FPS, model size. Latency is measured after warm-up with CUDA synchronisation, and FLOPs at a recorded input size (FLOPs without an input size are meaningless).

Watch the **FPS** column: the contribution is accuracy at controlled cost, not accuracy at any cost.

In [ ]:
for name, w in (('baseline', BASE_W), ('saryolo', OUR_W)):
    if w:
        !python -m saryolo efficiency --weights {w[0]} --imgsz {IMGSZ} --out results/efficiency/{name}

import json, glob
for f in sorted(glob.glob('results/efficiency/*/efficiency.json')):
    d = json.load(open(f))
    print(f"{f.split('/')[2]:<10} params={d.get('params_M')}M  GFLOPs={d.get('flops_G')}  FPS={d.get('fps')}  {d.get('latency_ms')}ms")

## 2. EXP-009 — Robustness

Controlled degradations: multiplicative Gamma speckle, contrast compression, blur, resolution loss, injected clutter. Deterministic given a seed, so both models face **byte-identical** inputs; only the images are degraded, since a physical degradation changes the sensor signal, not where the targets are.

Speckle is applied *multiplicatively* (the physically correct model for SAR), not as additive Gaussian noise.

In [ ]:
# `--limit` keeps the first pass cheap while you check the pipeline works.
if OUR_W:
    !python -m saryolo robustness --weights {OUR_W[0]} --data configs/datasets/{DATASET}.yaml --imgsz {IMGSZ} --out results/robustness --limit 50

In [ ]:
# Baseline sweep into the same folder, under the name the table generator expects.
import json, shutil
if BASE_W:
    !python -m saryolo robustness --weights {BASE_W[0]} --data configs/datasets/{DATASET}.yaml --imgsz {IMGSZ} --out results/robustness_baseline --limit 50
    src = 'results/robustness_baseline/robustness.json'
    if os.path.exists(src):
        shutil.copy(src, 'results/robustness/robustness_baseline.json')
        print('baseline robustness copied for comparison')

In [ ]:
from saryolo.paper import plot_robustness_curve
from IPython.display import Image, display
out = plot_robustness_curve(
    {'YOLO baseline': 'results/robustness/robustness_baseline.json',
     'SAR-YOLO': 'results/robustness/robustness.json'},
    'paper/figures/fig9_robustness.png')
if out:
    display(Image(str(out)))
else:
    print('No robustness data yet.')

## 3. EXP-011 — Cross-dataset generalization

Evaluates a checkpoint on a *different* dataset. The class-compatibility guard refuses to run when the label spaces do not line up, because a mAP computed over classes the model cannot predict is a number that measures nothing.

Prepare the second dataset with notebook 01 first (e.g. HRSID), then point `TARGET` at its config.

In [ ]:
TARGET = 'hrsid'   # set to None to skip
if TARGET and OUR_W and os.path.exists(f'datasets/processed/{TARGET}'):
    !python -m saryolo cross-dataset --weights {OUR_W[0]} --source configs/datasets/{DATASET}.yaml --target configs/datasets/{TARGET}.yaml --imgsz {IMGSZ} --out results/generalization
else:
    print('Skipped: prepare the target dataset first, or set TARGET = None.')

## 4. EXP-012 — Multi-seed stability

A single-seed difference of a few tenths of a point is not evidence. Repeats EXP-007 under different seeds so the headline result can be reported as mean ± std.

This is the most expensive optional step — run it only for the final chosen configuration.

In [ ]:
RUN_MULTISEED = False   # set True for the final configuration only
if RUN_MULTISEED:
    !python scripts/train_all_experiments.py --keep-going --only EXP-012

## 5. Qualitative figures

Ground truth vs predictions, Grad-CAM, and the failure taxonomy. `select_examples` picks the densest, sparsest and smallest-object scenes so the panels show the hard cases rather than flattering ones.

In [ ]:
from saryolo.training.trainer import load_model
from saryolo.visualization import comparison_panel, select_examples, save_attention_panel
from IPython.display import Image, display

if OUR_W:
    model = load_model(OUR_W[0])
    examples = select_examples(f'configs/datasets/{DATASET}.yaml', limit=3, mode='small')
    out = comparison_panel(model, examples,
                           labels_dir=f'datasets/processed/{DATASET}/labels/val',
                           out_path='paper/figures/fig7_qualitative.png', imgsz=IMGSZ, device='cuda')
    if out:
        display(Image(str(out)))
    if examples:
        panel = save_attention_panel(model, examples[0], 'paper/figures/fig8_attention.png', imgsz=IMGSZ, device='cuda')
        if panel:
            display(Image(str(panel)))

In [ ]:
from saryolo.evaluation.metrics import evaluate_detections, load_yolo_predictions, load_yolo_ground_truth
from saryolo.visualization.error_analysis import analyse_failures, image_contrast_map, write_failure_report

if OUR_W:
    metrics = evaluate_detections(OUR_W[0], f'configs/datasets/{DATASET}.yaml', imgsz=IMGSZ, out_dir='results/preds_failures')
    preds = load_yolo_predictions('results/preds_failures/labels', f'datasets/processed/{DATASET}/images/val')
    gts = load_yolo_ground_truth(f'configs/datasets/{DATASET}.yaml')
    report = analyse_failures(preds, gts, contrast_by_image=image_contrast_map(f'configs/datasets/{DATASET}.yaml'))
    print(report.summary())
    write_failure_report(report, 'results/failure_analysis.json')

## 6. Final paper assets

`--require-complete` is the submission gate: it raises if any cell is still `TBD`, so an unfinished table cannot be mistaken for a finished one.

In [ ]:
!python -m saryolo assets

from IPython.display import Markdown, display
import glob
for f in sorted(glob.glob('paper/tables/*.md')):
    display(Markdown('---'))
    display(Markdown(open(f).read()))

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/saryolo && cp -r results paper /content/drive/MyDrive/saryolo/
    print('saved results and paper assets to Drive')